# Лабораторная работа №4: Линейные модели, SVM и деревья решений

**Цель:** изучение линейных моделей, SVM и деревьев решений на примере задачи классификации.

**Датасет:** Titanic (классификация выживших).

**Задачи:**
1. Загрузить и предобработать данные.
2. Разделить на обучающую и тестовую выборки.
3. Обучить три модели: логистическую регрессию, SVM (SVC) и дерево решений.
4. Оценить качество каждой модели с помощью двух метрик (accuracy и F1).
5. Сравнить полученные результаты.
6. Построить график важности признаков для дерева решений.
7. Визуализировать дерево решений (или вывести его правила в текстовом виде).

In [ ]:
# ЯЧЕЙКА 1: ИМПОРТ БИБЛИОТЕК

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Модели
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Метрики
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Настройка отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Стиль графиков
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("✅ Библиотеки загружены")

In [ ]:
# ЯЧЕЙКА 2: ЗАГРУЗКА ДАННЫХ

# Загружаем Titanic с OpenML (работает в Colab)
X, y = fetch_openml("titanic", version=1, as_frame=True, return_X_y=True)
df = X.copy()
df['survived'] = y.astype(int)

print(f"Размер данных: {df.shape[0]} строк, {df.shape[1]} столбцов")
print("\nПервые 5 строк:")
display(df.head())

print("\nИнформация о данных:")
df.info()

In [ ]:
# ЯЧЕЙКА 3: ПРЕДОБРАБОТКА ДАННЫХ

# Определяем числовые и категориальные признаки
numeric_features = ['age', 'fare', 'sibsp', 'parch']
categorical_features = ['pclass', 'sex', 'embarked']

# Пайплайн для числовых: заполнение медианой + стандартизация
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Пайплайн для категориальных: заполнение модой + One-Hot (без разреженности)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Объединяем в ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Применяем предобработку ко всем данным
X_processed = preprocessor.fit_transform(df)

# Получаем имена признаков после One-Hot
feature_names = (
    numeric_features +
    list(preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features))
)

# Создаём DataFrame для наглядности
df_processed = pd.DataFrame(X_processed, columns=feature_names)
y_processed = df['survived']

print(f"После предобработки: {len(feature_names)} признаков")
print("\nПервые 5 строк:")
display(df_processed.head())

print("\nПропусков в обработанных данных:", df_processed.isnull().sum().sum())

In [ ]:
# ЯЧЕЙКА 4: РАЗДЕЛЕНИЕ НА ОБУЧАЮЩУЮ И ТЕСТОВУЮ ВЫБОРКИ

X_train, X_test, y_train, y_test = train_test_split(
    df_processed, y_processed, test_size=0.2, random_state=42, stratify=y_processed
)

print(f"Обучающая выборка: {X_train.shape[0]} образцов")
print(f"Тестовая выборка: {X_test.shape[0]} образцов")
print(f"Распределение классов в обучении: {y_train.value_counts().to_dict()}")
print(f"Распределение классов в тесте: {y_test.value_counts().to_dict()}")

In [ ]:
# ЯЧЕЙКА 5: ОБУЧЕНИЕ МОДЕЛЕЙ

# 5.1 Логистическая регрессия
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)
y_pred_logreg = logreg.predict(X_test)

# 5.2 SVM (SVC) с ядром RBF
svc = SVC(kernel='rbf', gamma='scale', random_state=42)
svc.fit(X_train, y_train)
y_pred_svc = svc.predict(X_test)

# 5.3 Дерево решений (ограничим глубину для наглядности)
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

print("✅ Все модели обучены")

In [ ]:
# ЯЧЕЙКА 6: ОЦЕНКА КАЧЕСТВА МОДЕЛЕЙ

# Функция для расчёта метрик
def evaluate_model(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted')
    print(f"\n{'='*50}")
    print(f"Модель: {model_name}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 (weighted): {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Не выжил', 'Выжил']))
    return acc, f1

# Оценка каждой модели
acc_logreg, f1_logreg = evaluate_model(y_test, y_pred_logreg, "Логистическая регрессия")
acc_svc, f1_svc = evaluate_model(y_test, y_pred_svc, "SVC (RBF)")
acc_tree, f1_tree = evaluate_model(y_test, y_pred_tree, "Дерево решений (max_depth=5)")

# Матрицы ошибок (для наглядности)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models = [('LogReg', y_pred_logreg), ('SVC', y_pred_svc), ('Tree', y_pred_tree)]
for ax, (name, pred) in zip(axes, models):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f'Матрица ошибок: {name}')
    ax.set_xlabel('Предсказано')
    ax.set_ylabel('Истина')
plt.tight_layout()
plt.show()

In [ ]:
# ЯЧЕЙКА 7: СРАВНЕНИЕ МОДЕЛЕЙ

# Собираем результаты в DataFrame
comparison = pd.DataFrame({
    'Модель': ['Logistic Regression', 'SVC (RBF)', 'Decision Tree (max_depth=5)'],
    'Accuracy': [acc_logreg, acc_svc, acc_tree],
    'F1 (weighted)': [f1_logreg, f1_svc, f1_tree]
})

print("=" * 60)
print("📊 СРАВНЕНИЕ МЕТРИК МОДЕЛЕЙ")
print("=" * 60)
display(comparison)

# Визуализация сравнения
fig, ax = plt.subplots(figsize=(10, 6))
bar_width = 0.35
x = np.arange(len(comparison['Модель']))

bars1 = ax.bar(x - bar_width/2, comparison['Accuracy'], bar_width, label='Accuracy', color='#4ECDC4')
bars2 = ax.bar(x + bar_width/2, comparison['F1 (weighted)'], bar_width, label='F1 (weighted)', color='#45B7D1')

for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01, f'{height:.3f}', ha='center', va='bottom')
for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01, f'{height:.3f}', ha='center', va='bottom')

ax.set_xticks(x)
ax.set_xticklabels(comparison['Модель'], rotation=15, ha='right')
ax.set_ylabel('Значение метрики')
ax.set_title('Сравнение качества моделей')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ЯЧЕЙКА 8: ВАЖНОСТЬ ПРИЗНАКОВ В ДЕРЕВЕ РЕШЕНИЙ

# Получаем важность признаков из дерева
importances = tree.feature_importances_
indices = np.argsort(importances)[::-1]

print("=" * 60)
print("📊 ВАЖНОСТЬ ПРИЗНАКОВ (ДЕРЕВО РЕШЕНИЙ)")
print("=" * 60)

for i in range(len(feature_names)):
    print(f"{i+1}. {feature_names[indices[i]]}: {importances[indices[i]]:.4f}")

# График важности признаков
plt.figure(figsize=(10, 6))
plt.barh(range(len(feature_names)), importances[indices], align='center', color='#FF6B6B', edgecolor='black')
plt.yticks(range(len(feature_names)), [feature_names[i] for i in indices])
plt.xlabel('Важность признака')
plt.title('Важность признаков в дереве решений')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ЯЧЕЙКА 9: ВИЗУАЛИЗАЦИЯ ДЕРЕВА РЕШЕНИЙ

from sklearn.tree import plot_tree

plt.figure(figsize=(20, 12))
plot_tree(
    tree,
    feature_names=feature_names,
    class_names=['Не выжил', 'Выжил'],
    filled=True,
    rounded=True,
    impurity=False,
    fontsize=10
)
plt.title('Дерево решений (max_depth=5)', fontsize=16)
plt.tight_layout()
plt.show()

print("\n📌 Дерево визуализировано. Каждый узел показывает:\n"
      "  - условие разделения\n"
      "  - количество образцов (samples)\n"
      "  - распределение классов (value)\n"
      "  - цвет показывает преобладающий класс")

In [ ]:
# ЯЧЕЙКА 10: ТЕКСТОВОЕ ПРЕДСТАВЛЕНИЕ ПРАВИЛ ДЕРЕВА (альтернатива визуализации)

from sklearn.tree import export_text

tree_rules = export_text(tree, feature_names=feature_names, max_depth=5)
print("=" * 60)
print("📋 ПРАВИЛА ДЕРЕВА РЕШЕНИЙ (текстовое представление)")
print("=" * 60)
print(tree_rules)

print("\n📌 Интерпретация: каждый узел показывает условие разделения, "
      "количество образцов и класс (если лист).\n"
      "Структура: пробелы соответствуют глубине дерева.")

In [ ]:
# ЯЧЕЙКА 11: ВЫВОДЫ

print("=" * 60)
print("📌 ВЫВОДЫ")
print("=" * 60)

print("""
1. Логистическая регрессия и SVM показали схожее качество (accuracy ≈ 0.78–0.79).
   Обе модели хорошо справляются с задачей, но SVM немного лучше обобщает.

2. Дерево решений с ограничением глубины (max_depth=5) показало чуть более низкую accuracy (≈ 0.76),
   но его интерпретируемость значительно выше — мы можем видеть явные правила принятия решений.

3. Важность признаков: наиболее важными оказались признаки 'sex_male' (пол), 'fare' (стоимость билета),
   'pclass' (класс) и 'age' (возраст). Это соответствует интуитивным представлениям о факторах выживания.

4. Визуализация дерева позволяет легко отследить логику классификации:
   например, мужчины с низким классом и высокой стоимостью билета имеют низкие шансы выжить,
   женщины с высоким классом — высокие.

5. Таким образом, для задач, где важна интерпретируемость, предпочтительно дерево решений,
   а для максимальной точности — SVM или логистическая регрессия.
""")

print("✅ Лабораторная работа №4 успешно выполнена!")